# Journal Entry Sentiment Analyzer (RAG)

For each journal entry: detect its emotion, retrieve similar **past** entries with FAISS, and generate a short reflection with FLAN-T5 grounded in those entries.

## 1. Imports and config

In [ ]:
!pip install faiss-cpu

In [35]:
import re
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
from transformers import pipeline

TOP_K = 3                     # past entries retrieved per query
N_REPORT = 100                 # entries to write reflections for (FLAN-T5 is slow on CPU); None = all

## 2. Load the dataset

In [36]:
import kagglehub, os
path = kagglehub.dataset_download("rtatman/blog-authorship-corpus")
print(os.listdir(path))
DATA_PATH = os.path.join(path, "blogtext.csv")
N_BLOGGERS = 50
MIN_POSTS = 30
MAX_POSTS = 100   # cap per blogger so a few prolific writers don't dominate

raw = pd.read_csv(DATA_PATH)
print(raw.columns.tolist(), raw.shape)

raw["date"] = pd.to_datetime(raw["date"].str.strip(), format="%d,%B,%Y", errors="coerce")
raw = raw.dropna(subset=["date", "text"])   # drops non-English month names too

counts = raw["id"].value_counts()
eligible = counts[counts >= MIN_POSTS].index
chosen = pd.Series(eligible).sample(n=min(N_BLOGGERS, len(eligible)), random_state=42)

df = (raw[raw["id"].isin(chosen)]
      .sort_values(["id", "date"])
      .groupby("id").head(MAX_POSTS)
      .reset_index(drop=True))

Using Colab cache for faster access to the 'blog-authorship-corpus' dataset.
['blogtext.csv']
['id', 'gender', 'age', 'topic', 'sign', 'date', 'text'] (681284, 7)


## 3. Preprocessing

Only whitespace is normalised. Punctuation and casing are kept because the emotion model and the sentence encoder were trained on natural text.

In [37]:
def clean_text(text):
    text = re.sub(r"\burllink\b", " ", str(text), flags=re.IGNORECASE)
    return re.sub(r"\s+", " ", text).strip()

df["cleaned_text"] = df["text"].apply(clean_text)
df = df[df["cleaned_text"].str.split().str.len() >= 30].reset_index(drop=True)   # drop very short posts

print(df["id"].nunique(), "bloggers |", len(df), "posts")

50 bloggers | 2553 posts


## 4. Load models

In [38]:
sentiment_pipeline = pipeline("text-classification",
                              model="j-hartmann/emotion-english-distilroberta-base")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
chat = pipeline("text-generation", model="Qwen/Qwen2.5-3B-Instruct", device_map="auto")

def llm(prompt, max_new_tokens=120, **kwargs):
    messages = [{"role": "user", "content": prompt}]
    out = chat(messages, max_new_tokens=max_new_tokens, do_sample=False)
    return [{"generated_text": out[0]["generated_text"][-1]["content"]}]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

## 5. Emotion analysis (once per entry)

Emotions are computed once for the whole dataset instead of being recomputed for every retrieved entry inside the loop. `truncation=True` stops long entries from crashing the 512-token model.

In [39]:
results = sentiment_pipeline(df["cleaned_text"].tolist(), truncation=True, batch_size=32)
df["emotion"] = [r["label"] for r in results]
df["emotion"].value_counts()

,count
emotion,
sadness,633
surprise,549
neutral,452
joy,389
disgust,206
fear,172
anger,152


## 6. Embeddings and FAISS index

Embeddings are L2-normalised and stored in an inner-product index, so search scores are cosine similarities.

In [40]:
embeddings = embedding_model.encode(
    df["cleaned_text"].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype(np.float32)

faiss_index = faiss.IndexFlatIP(embeddings.shape[1])
faiss_index.add(embeddings)
print("Indexed entries:", faiss_index.ntotal)

Batches:   0%|          | 0/80 [00:00<?, ?it/s]

Indexed entries: 2553


## 7. Retrieval

Two rules make the context meaningful:
- an entry never retrieves **itself** (it is always its own best match)
- only entries written **before** it are used, so a reflection never draws on the future

In [47]:
MIN_SIM = 0.40
authors = df["id"].to_numpy()
dates = df["date"].to_numpy()

def retrieve_similar_entries(idx, top_k=TOP_K):
    """Top-k most similar earlier posts by the SAME blogger."""
    scores, indices = faiss_index.search(embeddings[idx:idx + 1], faiss_index.ntotal)
    scores, indices = scores[0], indices[0]

    mask = (dates[indices] < dates[idx]) & (authors[indices] == authors[idx]) & (scores >= MIN_SIM)
    sel_i, sel_s = indices[mask][:top_k], scores[mask][:top_k]
    if len(sel_i) == 0:
        return df.iloc[0:0].assign(similarity=[])

    rows = df.iloc[sel_i].copy()
    rows["similarity"] = np.round(sel_s, 3)
    return rows

# Sanity check on one entry
i = 100
print("QUERY:", df.at[i, "cleaned_text"][:200], "|", df.at[i, "emotion"])
retrieve_similar_entries(i)[["date", "emotion", "similarity", "cleaned_text"]]

QUERY: hoguester's weblog Although I take offence to the 'Christian Bloggers' link at the top of this page and the fact that this little blogger felt it necessary to add a 'ster' to the end of Hogue, these ' | joy


,date,emotion,similarity,cleaned_text
88,2003-09-24,anger,0.576,Wahey! And now I have this super froody link d...
63,2003-07-24,surprise,0.543,Is it bad that my blog is completely about peo...
89,2003-09-24,joy,0.493,And now I've just signed up for this cool litt...


## 8. Retrieval check

If retrieval works, retrieved past entries should share the query's emotion more often than randomly chosen past entries.

In [48]:
rng = np.random.default_rng(42)
positions = np.arange(len(df))
retrieved_agree, random_agree = [], []

for idx in range(len(df)):
    past_idx = np.where((authors == authors[idx]) & (dates < dates[idx]))[0]
    if len(past_idx) == 0:
        continue
    similar = retrieve_similar_entries(idx)
    if len(similar) == 0:
        continue
    target = df.at[idx, "emotion"]
    retrieved_agree.append((similar["emotion"] == target).mean())
    rand_idx = rng.choice(past_idx, size=min(TOP_K, len(past_idx)), replace=False)
    random_agree.append((df.iloc[rand_idx]["emotion"] == target).mean())

print(f"Queries evaluated: {len(retrieved_agree)}")
print(f"Emotion agreement, retrieved past posts: {np.mean(retrieved_agree):.3f}")
print(f"Emotion agreement, random past posts:    {np.mean(random_agree):.3f}")

Queries evaluated: 1418
Emotion agreement, retrieved past posts: 0.348
Emotion agreement, random past posts:    0.297


## 9. Reflection generation

The instruction goes **first** in the prompt. FLAN-T5 truncates input at 512 tokens, so an instruction placed after a long entry gets cut off and the model just echoes the entry back. Retrieved entries are now actually included in the prompt, which is what makes this RAG.

In [43]:
def build_prompt(entry, emotion, similar, today_date):
    rules = (
        "You are a supportive assistant that reads a person's personal blog posts. "
        "Write a reflection of 2-3 sentences addressing the writer as 'you' (never 'I' or 'we'). "
        "Do not repeat the post, do not start with 'Reflecting', and use only details stated in the posts. "
        "If the post is not about the writer's own life or feelings (for example a joke, lyrics or news), "
        "write one neutral sentence about what they shared instead.\n\n"
    )
    today = f"Today's emotion: {emotion}\nToday's post: {entry[:800]}\n\n"

    if len(similar) == 0:
        return rules + today + "There are no related earlier posts, so focus only on today.\n\nReflection:"

    past = "\n".join(
        f"- {(today_date - r.date).days} days ago ({r.emotion}): {r.cleaned_text[:250]}"
        for r in similar.itertuples()
    )
    return (rules + today
            + f"Related earlier posts by the same writer:\n{past}\n\n"
            + "Briefly connect today's feeling to these earlier posts.\n\nReflection:")

def generate_reflection(idx):
    similar = retrieve_similar_entries(idx)
    prompt = build_prompt(df.at[idx, "cleaned_text"], df.at[idx, "emotion"],
                          similar, df.at[idx, "date"])
    out = llm(prompt, max_new_tokens=180, num_beams=4,
              no_repeat_ngram_size=3, truncation=True)[0]["generated_text"]
    if "." in out:
        out = out[: out.rfind(".") + 1]
    return out.strip(), similar

In [44]:
def copy_ratio(reflection, entry, n=5):
    """Fraction of the reflection's n-grams that appear word-for-word in the entry."""
    ref, ent = reflection.lower().split(), entry.lower().split()
    if len(ref) < n:
        return 0.0
    ref_grams = {tuple(ref[i:i + n]) for i in range(len(ref) - n + 1)}
    ent_grams = {tuple(ent[i:i + n]) for i in range(len(ent) - n + 1)}
    return len(ref_grams & ent_grams) / len(ref_grams)

copy_scores = []

## 10. Report

In [45]:
history = df.groupby("id").cumcount()   # how many earlier posts each post has
report_idx = (df.index[history >= 5].to_series()
              .sample(n=N_REPORT, random_state=42).sort_values().tolist())
lines = ["Journal Sentiment Analysis Report", "=" * 50]

for idx in report_idx:
  try:
        reflection, similar = generate_reflection(idx)
        sources = [df.at[idx, "cleaned_text"]] + similar["cleaned_text"].tolist()
        copy_scores.append(max(copy_ratio(reflection, s) for s in sources))

        past_emotions = ", ".join(similar["emotion"]) if len(similar) else "none"
        lines += [
            f"Blogger: {df.at[idx, 'id']}",
            f"Date: {df.at[idx, 'date'].date()}",
            f"Entry: {df.at[idx, 'cleaned_text']}",
            f"Emotion: {df.at[idx, 'emotion']}",
            f"Similar past entries: {len(similar)} ({past_emotions})",
            f"Reflection: {reflection}",
            "-" * 50,
        ]
        print(f"[{idx}] {df.at[idx, 'emotion']}: {reflection}")
  except Exception as e:
        print(f"Error on entry {idx}: {e}")

report = "\n".join(lines)
with open("sentiment_analysis_report.txt", "w", encoding="utf-8") as f:
    f.write(report)
print("Report saved: sentiment_analysis_report.txt")

[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[34] neutral: Reflecting on your journey through time, it seems you carried within you a curiosity for adventure and a deep-seated desire to understand the world, traits that have likely shaped your approach to various roles in your current life.


[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[54] fear: Today's post vividly captures your adventurous spirit and sense of humor, even amidst the fear of meeting George W. Bush and the intimidating drink cart lady.


[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[2441] surprise: Reflecting on your recent exploration of the concept of 'first true love,' it's clear that your curiosity about this enduring emotion has deepened, even if your initial skepticism remains. Your journey through research seems to have resonated with the sadness you felt after sharing a similar sentiment in an earlier post, where you pondered the impact of love on others.


[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[2467] disgust: It sounds like your routine has become particularly grueling today, given the heat and the additional responsibilities, contributing to your overall sense of frustration and discomfort.


[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[2518] fear: Reflecting on your post today, it seems you've shared a somber and concerning piece about potential future terrorist threats, highlighting the fear and uncertainty that such information can evoke.
[2540] neutral: Your recent post reflects the ongoing excitement and slight confusion as you navigate the process of choosing a color for Ria's nursery, which aligns with your previous posts about preparing her room and the uncertainty surrounding the final choice.
Report saved: sentiment_analysis_report.txt


In [46]:
copy_scores = np.array(copy_scores)
print(f"Reflections scored:  {len(copy_scores)}")
print(f"Mean copy ratio:     {copy_scores.mean():.3f}")
print(f"Copy rate (>50%):    {(copy_scores > 0.5).mean():.1%}")

Reflections scored:  100
Mean copy ratio:     0.000
Copy rate (>50%):    0.0%
